In [1]:
import pandas as pd
from implicit.nearest_neighbours import CosineRecommender, TFIDFRecommender
from rectools.dataset import Dataset
from rectools import Columns
from models.userknn import UserKnn
from rectools.model_selection.time_split import TimeRangeSplitter
from config.config_models import UserKnn_model_conf
from rectools.dataset import Interactions, Dataset


In [2]:
interactions = pd.read_csv('../data/kion_train/interactions.csv'.format(UserKnn_model_conf.dataset_path))
users = pd.read_csv('../data/kion_train/users.csv'.format(UserKnn_model_conf.dataset_path))
items = pd.read_csv('../data/kion_train/items.csv'.format(UserKnn_model_conf.dataset_path))


In [3]:
interactions.rename(columns={'last_watch_dt': Columns.Datetime,
                            'total_dur': Columns.Weight},
                    inplace=True)

interactions['datetime'] = pd.to_datetime(interactions['datetime'])


In [4]:
n_folds = UserKnn_model_conf.n_folds
unit = UserKnn_model_conf.unit
n_units = UserKnn_model_conf.n_units
freq = f"{n_units}{unit}"

# generator of folds
splitter = TimeRangeSplitter(
    test_size=freq,
    n_splits=n_folds,
    filter_already_seen=True,
    filter_cold_items=True,
    filter_cold_users=True,
)


In [5]:
(train_ids, test_ids, fold_info) = next(splitter.split(Interactions(interactions), collect_fold_stats=True))

In [6]:
train = interactions.loc[train_ids]
test = interactions.loc[test_ids]

In [7]:


model = UserKnn_model_conf.model

userknn_model = UserKnn(model=model, N_users=50)
userknn_model.fit(train)


/Users/rustemkhakim/recsys_ai_talent_hub/.venv/lib/python3.9/site-packages/implicit/utils.py:164: ParameterWarning: Method expects CSR input, and was passed coo_matrix instead. Converting to CSR took 0.12045121192932129 seconds
  warnings.warn(


  0%|          | 0/15480 [00:00<?, ?it/s]

In [ ]:
userknn_reco_df = userknn_model.predict(test, UserKnn_model_conf.N_recs)

In [9]:
userknn_reco_df.to_csv(UserKnn_model_conf.save_reco_df_path, encoding='utf-8', index=False)


NameError: name 'userknn_reco_df' is not defined

In [42]:
import dill

with open(UserKnn_model_conf.weight_path, 'wb') as f:
    dill.dump(userknn_model, f)


FileNotFoundError: [Errno 2] No such file or directory: 'data/weights/userknn_TFIDF.dill'